In [1]:
import time
from functools import wraps


def ttl_cache(seconds):
    """
    Caches function results for a given number of seconds using time.monotonic().
    
    Unhashable inputs behavior:
    If positional or keyword arguments contain unhashable types (such as lists or dicts),
    the call bypasses the cache, increments the miss counter, and executes the function directly.
    """
    if type(seconds) not in (int, float) or seconds <= 0:
        raise ValueError("TTL seconds positive number hona chahiye")

    def decorator(func):
        cache = {}
        stats = {"hits": 0, "misses": 0}

        @wraps(func)
        def wrapper(*args, **kwargs):
            # Keyword arguments ko sort karein taake x=2, y=3 aur y=3, x=2 same key banayein
            sorted_kwargs = tuple(sorted(kwargs.items()))
            key = (args, sorted_kwargs)

            # Check karein ke key hashable hai ya nahi (list/dict jaise unhashable inputs ke liye)
            try:
                hash(key)
            except TypeError:
                stats["misses"] += 1
                return func(*args, **kwargs)

            now = time.monotonic()

            # Agar key cache mein hai aur abhi expire nahi hui
            if key in cache:
                result, expiry_time = cache[key]
                if now < expiry_time:
                    stats["hits"] += 1
                    return result
                else:
                    # Expire ho chuki entry ko hata dein
                    del cache[key]

            # Cache miss: function ko run karke naya result store karein
            stats["misses"] += 1
            result = func(*args, **kwargs)
            cache[key] = (result, now + seconds)
            return result

        # Hits aur misses check karne aur cache clear karne ke liye helper functions
        def cache_info():
            return {"hits": stats["hits"], "misses": stats["misses"], "size": len(cache)}

        def clear_cache():
            cache.clear()
            stats["hits"] = 0
            stats["misses"] = 0

        wrapper.cache_info = cache_info
        wrapper.clear_cache = clear_cache
        return wrapper

    return decorator


# Testing
if __name__ == "__main__":
    @ttl_cache(seconds=2)
    def multiply(x, y):
        print("Computing...")
        return x * y

    print(multiply(x=2, y=3))       # Miss -> 6
    print(multiply(y=3, x=2))       # Hit -> 6 (keyword order different, still a hit)
    print("Stats:", multiply.cache_info())

    time.sleep(2.1)
    print(multiply(x=2, y=3))       # Miss (expired) -> 6
    print("Stats after expiry:", multiply.cache_info())

Computing...
6
6
Stats: {'hits': 1, 'misses': 1, 'size': 1}
Computing...
6
Stats after expiry: {'hits': 1, 'misses': 2, 'size': 1}
